# Validating CoLoRe simulation for Quasar Auto and Quasar x CMB

- The first and second CoLoRe simulation boxes were generated by Laura Casas, Dani L. Jarque run the rest from then on
- The notebook uses the sources output from CoLoRe, i.e. quasars, to validate the simulation through the auto-spectrum and cross spectrum with CMB lensing.
- The kappa values can be found in `./data/...test_box/`, along with $C_{\ell}$ (and its associated errors), and a logarithmic rebinning run with `./scripts/rebin_cls_qq.py`, which is the one used in this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import camb
from astropy.io import fits
import healpy as hp

from scipy.stats import chi2
from scipy.interpolate import CubicSpline
from scipy.integrate import quad, simpson
from scipy.constants import c
import sys
sys.path.append('../')

from py.rebinning import log_bin_cls  
import json

plt.rcParams.update({
    "text.usetex": True,
    "font.family": 'serif',
})

# Theory CMB lensing auto-spectrum $C_{\ell}^{\kappa \kappa}$

## setting up CAMB

In [ ]:
def load_cosmology(filename):
    with open(filename) as f:
        return json.load(f)

In [ ]:
def get_camb_results(
    cosmo_dict,
    z_pk,
    kmax=100.0,
    lmax=2500,
    nonlinear=False,
    window=False,
):
    """
    Read a cosmological parameter dictionary, extract cosmological parameters,
    and build:

    1. CAMBparams instance
    2. CAMB results object

    Parameters
    ----------
    cosmo_dict : dict
        Dictionary containing cosmological parameters

    z_pk : list or None
        Redshifts for P(k) interpolator.
        If None, defaults to [0.0]

    kmax : float
        Maximum k [1/Mpc]

    lmax : integer
        Maximum multipole

    nonlinear : bool
        Whether to use nonlinear P(k)
    """

    # --------------------------------------------------
    # Convert to CAMB inputs
    # --------------------------------------------------

    H0 = 100.0 * cosmo_dict["h"]
    ombh2 = cosmo_dict["omega_B"] * cosmo_dict["h"]**2
    omch2 = (cosmo_dict["omega_M"] - cosmo_dict["omega_B"]) * cosmo_dict["h"]**2

    # --------------------------------------------------
    # Initial CAMB params
    # --------------------------------------------------

    pars = camb.CAMBparams()

    pars.set_cosmology(
        H0=H0,
        ombh2=ombh2,
        omch2=omch2,
    )

    pars.set_dark_energy(w=cosmo_dict["w"])
    pars.InitPower.set_params(
        ns=cosmo_dict["ns"],
        As=cosmo_dict["As"]
    )

    # --------------------------------------------------
    # Matter power settings
    # --------------------------------------------------

    pars.set_matter_power(
        redshifts=z_pk,
        kmax=kmax
    )

    # note that to get CAMB's internal calculation accurate at the 1% level at L~2000,
    # need lens_potential_accuracy=2. Increase to 4 for accurate match to the Limber calculation here
    if nonlinear:
        pars.NonLinear = camb.model.NonLinear_pk
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=True)
    else:
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=4, nonlinear=False)


    # Final results
    results = camb.get_results(pars)

    return results

## manual Limber integral

In [ ]:
def w_qq(z, bz_interp, nz_interp):
    '''
    CMB lensing convergence kernel
    '''

    # evaluate bias and redshift distribution at z
    bias = bz_interp(z)
    ndensity = nz_interp(z)
    
    return bias * ndensity

In [ ]:
def limber(z, ell, kmax, bz_interp, nz_interp, pk_interp, results):
    '''
    Quasar auto-spectrum using Limber approximation
    '''

    chi = results.comoving_radial_distance(z) # converting chi to redshift
    k = (ell + 0.5) / chi  # Limber relation
    c_kms = c*1E-3  # convert speed of light to km/s for consistency with CAMB outputs in Mpc/h and km/s
    Hz = results.hubble_parameter(z) # Hubble parameter at redshift z in km/s/Mpc

    kp = 0.312  # (1/Mpc) k pressure parameter scale used for smoothing the power spectrum according to the colore simulation resolution

    # kernel and pk interpolator, filtering out of extrapolation k and small k
    if 1e-4 < k < kmax:
        kernel = w_qq(z, bz_interp, nz_interp)
        pk = pk_interp.P(z, k, grid=False)
        smooth = np.exp(-(k/kp)**2)  # smoothing factor to avoid oscillations at high k 
        return (Hz/c_kms) * (kernel**2 / chi**2) * pk * smooth
        
    else:
        print(ValueError(f"Warning: k={k:.3e} is out of bounds for the power spectrum interpolation. Check the range of ell and redshift, and the kmax used for CAMB."))
        return 0

In [ ]:
def get_limber_cls_qq(cosmo_params_filename, z_pk, bz_interp, nz_interp, nside, kmax=100, lmin=2, lmax=2500):
    '''
    Compute the quasar auto-spectrum using the Limber approximation, integrating up to z_max from the cosmology file.
    '''

    # get linear and non-linear matter power spectrum
    cosmo_dict = load_cosmology(cosmo_params_filename)
    z_max, z_out = cosmo_dict["z_max"], cosmo_dict["z_out"]

    results_lin = get_camb_results(
        cosmo_dict=cosmo_dict,
        z_pk=z_pk,
        kmax=kmax,
        lmax=lmax,
        nonlinear=False)

    # get the matter power spectrum interpolators for linear and non-linear cases
    pk_interp_lin = results_lin.get_matter_power_interpolator(nonlinear=False, 
                                                      hubble_units=False, 
                                                      k_hunit=False,  
                                                      var1='delta_tot', 
                                                      var2='delta_tot')
    
    #store the cls from linear and nonlinear limber approximation integration
    # redshift integration up to zmax
    ells = np.arange(lmin, lmax+1)
    cl_qq_lin_zmax_zs = np.zeros(ells.shape)

    # manually integrate the limber approximation
    for i, ell in enumerate(ells):
        
        # integral up to redshift z_max, source plane at z_star
        z_cut = z_max
        #print(f"Integrating for ell={ell} with z_cut={z_cut:.2f}")
        cl_qq_lin_zmax_zs[i] = quad(limber, 1e-4, z_cut, args=(ell, kmax, bz_interp, nz_interp, pk_interp_lin, results_lin))[0]

    lmax_map = nside * 3 - 1  # maximum multipole for a HEALPix map at given nside
    pixwin = hp.pixwin(nside, lmax=lmax_map)  # pixel window function for HEALPix map at given nside and lmax NSIDE*3 - 1
    cl_qq_lin_zmax_zs_smoothed = cl_qq_lin_zmax_zs * pixwin[ells]**2  # deconvolve the pixel window function

    return {
        'ells': ells,
        'results_lin': results_lin,
        'cl_qq_lin_zmax_zs': cl_qq_lin_zmax_zs_smoothed,
    }


In [ ]:
# set the redshift range for the boltzmann solver
zmin, zmax = 0, 1100
nz = 200
z_pk = zmin + np.exp(np.log(zmax - zmin + 1)*np.linspace(0, 1, nz)) - 1
z_pk = z_pk[::-1] # reverse to have higher redshift first for better interpolation at high z
lmin, lmax = 2, 2500
nside = 512

In [ ]:
# load the quasar bias and redshift distribution, and create interpolators for them
z_bz, bz = np.loadtxt('../data/colore_files/Bz_qso_G18.txt', unpack=True)
z_nz, nz = np.loadtxt('../data/colore_files/Nz_qso_130618_2_colore1_hZs.txt', unpack=True)
norm = simpson(nz, z_nz)
nz_norm = nz/norm
nz_interp = CubicSpline(z_nz, nz_norm, extrapolate=False)
bz_interp = CubicSpline(z_bz, bz, extrapolate=False)

In [ ]:
# get the limber cls for the second and third boxes
box3 = get_limber_cls_qq('../data/third_test_box/cosmo_params.json', z_pk=z_pk, kmax=100, 
                         lmin=lmin, lmax=lmax, bz_interp=bz_interp, nz_interp=nz_interp, nside=nside)

# extract the relevant results for the second and third boxes
keys = ['ells', 'cl_qq_lin_zmax_zs']

ells, cl_qq_lin_zmax_zs_box3 = [box3[k] for k in keys]

# CoLoRe simulation auto-spectrum $C_{\ell}^{q q}$

In [ ]:
def merge_quasar_catalog(quasar_folder):
    '''
    Merge the quasar catalogs from the different HEALPix tiles into a single catalog of RA and DEC.
    The catalogs are assumed to be named out_srcs_s1_i.fits, where i is the tile number.
    '''

    ra_list = []
    dec_list = []
    
    i = 0
    # Loop over the HEALPix tiles until we run out of files
    while True:
        try:
            cat = fits.getdata(f'{quasar_folder}out_srcs_s1_{i}.fits', ext=1)
        except FileNotFoundError:
            break
        ra_list.append(cat['RA'])
        dec_list.append(cat['DEC'])
        i += 1
        ra_list.append(cat['RA'])
        dec_list.append(cat['DEC'])

    RA = np.concatenate(ra_list)
    DEC = np.concatenate(dec_list)

    print(f'Merged {i} catalogs from {quasar_folder} into a single catalog with {len(RA)} quasars.')

    return RA, DEC

In [ ]:
def compute_cls_qq(quasars_folder, nside):

    ra, dec = merge_quasar_catalog(quasars_folder)

    # Choose resolution
    nside = nside  # resoluition parameter for HEALPix map, nside=4096 corresponds to ~1 arcmin pixels

    # Convert coordinates
    theta = np.radians(90.0 - dec)   # colatitude
    phi   = np.radians(ra)  # longitude

    # Pixel indices
    pix = hp.ang2pix(nside, theta, phi)

    # Count quasars per pixel
    npix = hp.nside2npix(nside)
    counts = np.bincount(pix, minlength=npix)
    
    # Mean counts per pixel for pixel
    mean_counts = np.mean(counts)
    print(f"Mean counts per pixel: {mean_counts:.2f}")

    # Compute the overdensity map (delta) for pixels with counts > 0, and set delta=0 for empty pixels to avoid division by zero
    delta = (counts - mean_counts) / mean_counts
    lmax_map = hp.npix2nside(npix) * 3 - 1  # maximum multipole for the map, set by the HEALPix resolution
    cl = hp.anafast(delta, lmax=lmax_map)  # compute the auto power spectrum of the overdensity map

    # compute the effective sky fraction and number density for shot noise calculation
    fsky = 1.0  # full sky so far, otherwise 1.0 * np.sum(mask) / npix
    nbar_sr = len(ra) / (4*np.pi*fsky)
    shot_noise = 1.0 / nbar_sr

    # subtract shot noise from the auto-spectrum and deconvolve the pixel window function to get an estimate of the underlying quasar clustering power spectrum
    cl_signal = cl - shot_noise

    return cl_signal, cl, fsky, shot_noise

In [ ]:
def save_rebinned_cls_qq(quasars_folder, nbins, out_dir):
    '''
    Read the quasar maps from the specified folder, compute the power spectrum, rebin it into logarithmic bins, and save the rebinned Cls and their errors to .npy files.
    '''

    # Read the quasar power spectrum and compute the error bars assuming Gaussianity and no CMB lensing noise
    cl_qq, cl, fsky, shot_noise = compute_cls_qq(quasars_folder)
    ell = np.arange(len(cl_qq))  # corresponding multipole moments
    err_tt = np.sqrt(2/ ((2*ell + 1)*fsky) ) + (cl + shot_noise)**2   # error on the Cls, assuming Gaussianity
    ell_colore_log, cl_qq_log, err_tt_log = log_bin_cls(ell, cl_qq, nbins=nbins)

    # Save the rebinned Cls and their errors to .npy files
    np.savez(f'{out_dir}/cl_qq', ells=ell, cls=cl_qq, err=err_tt)
    np.savez(f'{out_dir}/cl_qq_log', ells=ell_colore_log, cls=cl_qq_log, err=err_tt_log)

In [ ]:
cl_qq_signal, cl_qq, fsky, shot_noise = compute_cls_qq('../data/third_test_box/quasars/', nside=nside)

In [ ]:
nbins = 20
ells_colore, cl_qq_signal_rebin, err_tt_signal_rebin = log_bin_cls(np.arange(len(cl_qq_signal)), cl_qq_signal, nbins=nbins, ell_min=lmin, ell_max=lmax)
ells_colore, cl_qq_rebin, err_tt_rebin = log_bin_cls(np.arange(len(cl_qq)), cl_qq, nbins=nbins, ell_min=lmin, ell_max=lmax)
mask = np.isfinite(ells_colore)

In [ ]:
ells_colore = ells_colore[mask]
cl_qq_signal_rebin = cl_qq_signal_rebin[mask]
err_tt_signal_rebin = err_tt_signal_rebin[mask]
cl_qq_rebin = cl_qq_rebin[mask]
err_tt_rebin = err_tt_rebin[mask]

## plotting

## latest CoLoRe box analysis and $\chi^2$ test

In [ ]:
# rebin the CAMB linear and non-linear cls to match the number of bins in the Colore data for residuals
cl_qq_lin_zmax_zs_box3_rebin = log_bin_cls(ells, cl_qq_lin_zmax_zs_box3, nbins=nbins, ell_min=lmin, ell_max=lmax)[1]

# filter out NaN values which may arise from empty bins in the rebinning process

cl_qq_lin_zmax_zs_box3_rebin = cl_qq_lin_zmax_zs_box3_rebin[mask]

In [ ]:
def plot_cls_comparison(ells_colore, cl_qq_colore_signal, cl_qq_colore, err_tt_signal, err_tt, ells, cl_qq_lin, cl_qq_lin_rebin, box_label):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), height_ratios=[2.5, 1], dpi=120, sharex=True)

    # compute the reduced chi-squared between the rebinned linear limber and the rebinned CoLoRe cls
    chi2 = np.sum((cl_qq_colore_signal - cl_qq_lin_rebin)**2 / err_tt_signal**2)
    chi2_nu = chi2 / (len(ells_colore))

    # plot the rebinned CoLoRe Cls with error bars
    ax1.errorbar(ells_colore, cl_qq_colore_signal, yerr=err_tt_signal, elinewidth=0.7, capsize=1.0, fmt='C0.', markersize=5.0, 
                 label=rf'CoLoRe_{box_label} (shot-noise subtraction)')
    ax1.errorbar(ells_colore, cl_qq_colore, yerr=err_tt, elinewidth=0.7, capsize=1.0, fmt='C1.', markersize=5.0, 
                 label=rf'CoLoRe_{box_label} (no shot-noise subtraction)')
    ax1.scatter(0, 0 , s=0, label=rf'$\chi^2_\nu = {chi2_nu:.2f}$')

    # manual limber integration full redshift
    ax1.plot(ells, cl_qq_lin, c='C0', ls='-', label=rf'Limber_linear')


    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlim(1.5, lmax)
    ax1.set_ylabel(r'$C_{L}^{qq}$')
    ax1.legend()

    # plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
    rel_error = (cl_qq_colore_signal - cl_qq_lin_rebin) / cl_qq_lin_rebin
    rel_error_err = err_tt_signal / cl_qq_lin_rebin

    # split the relative error plot into two panels for better visibility of low and high multipoles
    l_lim = ells_colore.max()

    # plot the relative error for low multipoles in the main panel
    ax2.errorbar(ells_colore[ells_colore < l_lim], rel_error[ells_colore < l_lim], 
                yerr=rel_error_err[ells_colore < l_lim], capsize=1.0, elinewidth=0.7, fmt='C0.', markersize=5.0)
    ax2.axhline(0, c='k', ls='--', lw=0.7)
    ax2.set_ylim(-1.0, 1.0)
    ax2.set_xscale('log')
    ax2.set_xlabel(r'Multipole $L$')
    ax2.set_ylabel(rf'rel. error')

    fig.suptitle('Quasar auto-spectrum: Theory vs. CoLoRe boxes', y=0.93)

    fig.savefig(f'plots/cls_qq_{box_label}.png', bbox_inches='tight')
    fig.savefig(f'plots/cls_qq_{box_label}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
def plot_cumulative_chi2(ells_colore, cl_kk_colore, err_tt, cl_kk_lin_rebin, box_label):

    # compute the cumulative chi-square and p-values for each multipole bin
    chi2_per_bin = ((cl_kk_colore - cl_kk_lin_rebin) / err_tt)**2
    cumulative_chi2 = np.nancumsum(chi2_per_bin)
    cumulative_dof = np.arange(1, len(cumulative_chi2) + 1)
    p_values = 1 - chi2.cdf(cumulative_chi2, df=cumulative_dof)

    # Plotting the cumulative chi-square
    fig, ax1 = plt.subplots(figsize=(6, 4), dpi=120)
    ax2 = ax1.twinx()  # Create a second y-axis for p-values

    # Plot your cumulative chi-square
    ax1.plot(ells_colore, cumulative_chi2, marker='.', linestyle='-', color='C0', label=rf'Cumulative $\chi^2$ ({box_label})')
    ax2.plot(ells_colore, p_values, marker='.', linestyle='-', color='C1', label=r'Cumulative $p$-value', alpha=0.3)

    # Plot the expected theoretical baseline (Expected chi^2 = dof)
    ax1.plot(ells_colore, cumulative_dof, linestyle='--', color='black', label=r'Expected $\chi^2$ (Perfect Fit)')

    # Formatting the plot
    ax1.set_xscale('log')
    ax1.set_yscale('linear') # A log scale on the y-axis is highly recommended since the max value is ~8000
    ax1.set_ylim(0, 50)
    ax1.set_xlabel('Multipole $L$')
    ax1.set_ylabel(r'Cumulative $\chi^2$')
    ax2.set_ylabel(r'Cumulative $p$-value')
    ax1.set_title(r'Cumulative $\chi^2$ vs. Multipole $L$')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')

    fig.savefig(f'plots/cls_qq_{box_label}_cumulative_chi2.png', bbox_inches='tight')  
    fig.savefig(f'plots/cls_qq_{box_label}_cumulative_chi2.pdf', bbox_inches='tight')  

    plt.show()

In [ ]:
plot_cls_comparison(ells_colore, cl_qq_signal_rebin, cl_qq_rebin, err_tt_signal_rebin, err_tt_rebin, ells, cl_qq_lin_zmax_zs_box3, 
                    cl_qq_lin_zmax_zs_box3_rebin, box_label='third_box')  